# 88 · Species differences carried by other paralogs

The 1:1 analysis pairs one human gene with one mouse gene. When the pair sits in a larger family, a human–mouse
difference has readings that only the rest of the family can separate:

| Part | Question |
|---|---|
| P1 | Substitution: does another member in the other species carry the profile the kept partner lacks? |
| P2 | Compensation: do the family sums agree where the kept pair differs? |
| P3 | Division of labour: does a single gene span what several genes split in the other species? |
| P4–P6 | Position vs sequence pairing; differences only family units see; niches of genes without an ortholog |

**Protocol.** `docs/results/pt-paralog-substitution.md` (frozen before any statistic). Curves are notebook 86's W
runs on PT-identity units (U_PT). Two male mice, one male human donor; descriptive of this cohort.

## 1 · Setup

In [ ]:
import argparse
import gzip
import hashlib
import json
import os
import re
import sys
import time
from pathlib import Path

for _var in ('OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS'):
    os.environ.setdefault(_var, '1')
NOTEBOOK_START = time.perf_counter()
starts = [Path.cwd()] + ([Path(__file__).resolve().parent] if '__file__' in globals() else [])
project = next(p for s in starts for p in (s, *s.parents) if (p / 'pseudospace').is_dir())
sys.path.insert(0, str(project))
parser = argparse.ArgumentParser(add_help=False)
parser.add_argument('--data-root', type=Path)
parser.add_argument('--results-root', type=Path)
args, _ = parser.parse_known_args()
data_root = Path(args.data_root or os.environ.get('PSEUDOSPACE_DATA_ROOT') or project / 'data').expanduser()
results_root = Path(args.results_root or os.environ.get('PSEUDOSPACE_RESULTS_ROOT') or project / 'results').expanduser()

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

PKG = results_root / 'package_pseudospace'
EXPORT = {'hvm': PKG / 'hvm' / 'pt_units.h5ad'}
STAGE01 = PKG / 'runs' / 'hvm' / '01_units'
upstream13 = results_root / 'minimal_pt_scfates'
T62 = results_root / 'pt_non_one_to_one_orthologs'               # read only: coordinate-free inventory columns
T71 = results_root / 'pt_pathways_package_pseudospace'           # read only: universe, caches, statistics
T72 = results_root / 'pt_package_coordinate_validation'          # read only: transitions
EXT = data_root / 'external'
COMPARA_HM = EXT / 'ensembl_compara_human_mouse' / 'Compara.98.protein_default.homologies.human_mouse.tsv.gz'
T86 = results_root / 'pt_non_one_to_one_package'          # read only: W runs, tables
output = results_root / 'pt_paralog_substitution'
tables, figures, cache = output / 'tables', output / 'figures', output / 'stage_cache'
for folder in (tables, figures, cache):
    folder.mkdir(parents=True, exist_ok=True)

DS = 'human_vs_mouse'
MOUSE, HUMAN = ['Ctrl1A2', 'Ctrl1A4'], ['HUK1_COR1', 'HUK1_MED1']
SPECIES = {'mouse': MOUSE, 'human': HUMAN}
REQUIRED = [EXPORT['hvm'], *[STAGE01 / f'{s}_units.h5ad' for s in MOUSE + HUMAN], upstream13 / 'ortholog_map_used.csv',
            T62 / 'tables' / 'gene_inventory_mouse.csv', T62 / 'tables' / 'gene_inventory_human.csv',
            T72 / 'tables' / 'transitions.csv', COMPARA_HM,
            *[T86 / 'tables' / f for f in ('w_calls_mouse.csv', 'w_calls_human.csv', 'h7_family_units.csv',
                                            'h7b_one_to_one_calls_in_families.csv')],
            *[T71 / 'tables' / f for f in ('inputs.csv', f'universe_{DS}.csv', f'gene_sets_{DS}.json', f'gene_statistics_{DS}.csv')]]
for path in REQUIRED:
    if not path.is_file():
        raise FileNotFoundError(f'Missing input: {path} (run notebooks 62, 70–72 and 86 first; see data/README.md)')

NOTEBOOK_LOGIC_VERSION = '88.paralog_substitution.1'
FINGERPRINT = {'hvm': 'efafc9ac2e99'}
N_JOBS = int(os.environ.get('PSEUDOSPACE_N_JOBS', '24'))
SEED = 88
# Protocol constants (docs/results/pt-paralog-substitution.md)
NON = ('1:many', 'many:1', 'many:many')
ALPHA = .05
GAIN, R_ALT = .5, .5          # P1: substituted
L_ACCOUNTED = -1.             # P2: the difference at least halves
N_NULL_P1, N_NULL_SUMS = 1000, 200
SEGMENTS = ('S1', 'S2', 'S3')

MM = 1 / 25.4
INK, LIGHT, GREY = '#0b0b0b', '#d9d8d4', '#9a9893'
CLASS_COLOR = {'1:1': '#9a9893', 'non-1:1': '#C0392B', 'no ortholog': '#7D3C98'}
SPEC_COLOR = {'Ctrl1A2': '#0072B2', 'Ctrl1A4': '#56B4E9', 'HUK1_COR1': '#D55E00', 'HUK1_MED1': '#E69F00'}
plt.rcParams.update({
    'font.family': ['Liberation Sans', 'DejaVu Sans'], 'font.size': 6.5, 'axes.titlesize': 6.5, 'axes.titleweight': 'bold',
    'axes.labelsize': 6.5, 'xtick.labelsize': 6, 'ytick.labelsize': 6, 'legend.frameon': False,
    'axes.spines.top': False, 'axes.spines.right': False, 'axes.linewidth': .6, 'lines.linewidth': 1.0,
    'pdf.fonttype': 42, 'svg.fonttype': 'none', 'savefig.dpi': 300})
pd.set_option('display.width', 240, 'display.max_columns', 40, 'display.max_colwidth', 70, 'display.max_rows', 200)
GUARDS, PRIMARY = [], []


def save(fig, stem):
    for suffix in ('png', 'pdf'):
        fig.savefig(figures / f'{stem}.{suffix}', bbox_inches='tight')
    plt.show()
    plt.close(fig)


def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''):
            h.update(chunk)
    return h.hexdigest()


def stored(folder, stage):
    """A notebook's newest cached payload of one stage (read only); raises when its cache was purged."""
    files = sorted((folder / 'stage_cache').glob(f'{stage}__*.npz'),
                   key=lambda p: json.loads(Path(str(p) + '.key.json').read_text())['written_utc'])
    if not files:
        raise FileNotFoundError(f'{folder.name}: no cached {stage}')
    with np.load(files[-1], allow_pickle=False) as payload:
        return {k: payload[k] for k in payload.files}


def primary(test, p, statistic, note=''):
    PRIMARY.append({'test': test, 'p': float(p), 'statistic': statistic, 'note': note})


print('Results folder:', output, '· logic', NOTEBOOK_LOGIC_VERSION)

## 2 · Inputs and guards (not cached)

G1 `hvm_inputs` reproduces 71; G2 72's transitions; G3 the stage-01 native panels match the export unit by unit;
G4 62's inventories list the panels. G5 (86's cached W runs) follows in section 3.

In [ ]:
import anndata as ad
from scipy import sparse

from pseudospace import non_one_to_one as no
from pseudospace import paralog_zonation as pz
from pseudospace import species_calls as sc
from pseudospace.continuous_descriptors import logistic_transitions
from pseudospace.package_inputs import hvm_inputs, read_pt_units
from pseudospace.pathway_pipelines import common_grid
from pseudospace.stage_cache import cached_payload, digest

t0 = time.perf_counter()
orthologs = pd.read_csv(upstream13 / 'ortholog_map_used.csv')
MT = sorted(g for g in orthologs.mouse_symbol.astype(str) if g.startswith('mt-'))
assert len(MT) == 13, MT
UNITS = read_pt_units(EXPORT['hvm'], 'hvm')
PROV = UNITS.uns['provenance']
assert str(PROV['notebook_logic_version']).startswith('70.'), PROV['notebook_logic_version']
assert PROV['fingerprints']['pass1']['membership_sha1'] == FINGERPRINT['hvm'], PROV['fingerprints']['pass1']
INP = hvm_inputs(UNITS, data_root, orthologs, exclude_genes=MT)
GENES71 = list(INP['genes'])
universe71 = pd.read_csv(T71 / 'tables' / f'universe_{DS}.csv')
assert GENES71 == universe71[universe71.eligible.astype(bool)].gene.astype(str).tolist(), 'G1: universe differs from 71'
assert INP['gene_sets'] == json.loads((T71 / 'tables' / f'gene_sets_{DS}.json').read_text()), 'G1: gene sets differ from 71'
assert int(pd.read_csv(T71 / 'tables' / 'inputs.csv', index_col=0).loc['units in the common support', 'Human vs mouse PT']) == len(INP['position'])
POSITION = np.asarray(INP['position'], float)
SPECIMEN = INP['specimen'].astype(str)
SEGMENT = np.asarray(INP['segment']).astype(str)
LIBRARY = np.asarray(INP['library'], float)
OBS = INP['obs']
IDENT = OBS['primary_identity'].astype(str).to_numpy()
COND71 = stored(T71, f'gene_model_{DS}_species')
STAT71 = pd.read_csv(T71 / 'tables' / f'gene_statistics_{DS}.csv', header=[0, 1], index_col=0)
np.testing.assert_allclose(COND71['stat|T_spatial'], STAT71['condition'].T_spatial.to_numpy(), rtol=1e-12, atol=0)
GRID, KNOTS = COND71['grid'], COND71['knots']
np.testing.assert_allclose(GRID, common_grid(POSITION, SPECIMEN, 101), rtol=0, atol=1e-12)
GUARDS.append(f'G1: {len(GENES71):,} genes and {len(POSITION):,} units equal 71; 71\'s cached condition model reproduces '
              f'its T_spatial; knots {np.round(KNOTS, 3).tolist()}')

TRANSITIONS = pd.read_csv(T72 / 'tables' / 'transitions.csv')
TRANSITIONS = TRANSITIONS[TRANSITIONS.dataset.eq('hvm')].reset_index(drop=True)
again_t = logistic_transitions(POSITION, SEGMENT, SPECIMEN).merge(TRANSITIONS, on=['specimen', 'transition'])
assert len(again_t) == 8 and np.allclose(again_t.position_x, again_t.position_y, atol=1e-6)
MOUSE_CUTS = sc.mean_transitions(TRANSITIONS, MOUSE)
GUARDS.append(f'G2: transitions reproduce 72\'s; mouse cuts {MOUSE_CUTS[0]:.3f}, {MOUSE_CUTS[1]:.3f}')


def to_mouse(position):
    return sc.to_segment_units(position, MOUSE_CUTS)


GRID_UNITS = {'mouse': to_mouse(GRID), 'human': GRID.copy()}

# G3: native panels, unit by unit
NATIVE, PANEL = {}, {}
for species, names in SPECIES.items():
    blocks = []
    for name in names:
        native = ad.read_h5ad(STAGE01 / f'{name}_units.h5ad')
        rows = np.flatnonzero(SPECIMEN == name)
        ids = OBS.polygon_id.astype(str).to_numpy()[rows]
        assert pd.Index(ids).isin(native.obs_names).all() and pd.Index(ids).is_unique, f'G3: {name} units missing'
        x = sparse.csr_matrix(native[list(ids)].X, dtype=float)
        export_x = sparse.csr_matrix(UNITS[OBS.index[rows]].X, dtype=float)
        symbols = UNITS.var_names if species == 'mouse' else pd.Index(UNITS.var.human_symbol.astype(str))
        cols = native.var_names.get_indexer(symbols)
        assert (cols >= 0).all(), f'G3: {name} lacks export genes'
        assert (x[:, cols] != export_x).nnz == 0, f'G3: {name} counts differ from the export'
        if species in PANEL:
            assert PANEL[species].equals(native.var_names)
        PANEL[species] = pd.Index(native.var_names.astype(str))
        blocks.append((rows, x))
    order = np.concatenate([r for r, _ in blocks])
    NATIVE[species] = {'rows': order, 'counts': sparse.vstack([x for _, x in blocks]).tocsr()}
GUARDS.append('G3: every export unit is in its stage-01 file; shared genes identical; panels mouse '
              f"{len(PANEL['mouse']):,}, human {len(PANEL['human']):,} genes")

INV_COLUMNS = ['gene', 'gene_id', 'component', 'n_human', 'n_mouse', 'ortholog_class', 'class_support2', 'class_support3',
               'class_support5', 'paper_partner', 'probe_status', 'offtarget_genes', 'probe_clean', 'probe_clean_strict', 'n_probes']
INV = {}
for species in SPECIES:
    inv = pd.read_csv(T62 / 'tables' / f'gene_inventory_{species}.csv', usecols=INV_COLUMNS)
    assert inv.gene.astype(str).tolist() == PANEL[species].tolist(), f'G4: {species} inventory differs from the panel'
    inv = inv.set_index('gene')
    inv['family_clean'] = no.family_clean(inv)
    inv['non'] = inv.class_support3.isin(NON)
    INV[species] = inv
GUARDS.append('G4: 62\'s inventories list the stage-01 panels in order (coordinate-free columns only)')

EXTERNAL_SHA = {str(COMPARA_HM.relative_to(data_root)): sha256(COMPARA_HM)}
print('\n'.join('· ' + g for g in GUARDS))
print(f'Inputs {time.perf_counter() - t0:.0f} s')

## 3 · Notebook 86's within-species curves (G5)

86's cached W runs on U_PT are read, not refitted. G5: the payloads list 86's PT-expressed genes, and refitting the
first 200 mouse genes with 86's typical shapes reproduces their centred curves.

In [ ]:
from scipy.stats import wilcoxon

U_PT = IDENT == 'Proximal_Tubule'
code_key = digest([NOTEBOOK_LOGIC_VERSION, Path(no.__file__).read_text()])
SECONDS = {}


def cached(stage, compute, *, params=None, inputs=None):
    def wrapped():
        t = time.perf_counter()
        out = dict(compute())
        out['__seconds'] = np.array(time.perf_counter() - t)
        return out
    result = cached_payload(stage, wrapped, root=cache, params=params, inputs=inputs, code=code_key)
    SECONDS[stage] = float(result.pop('__seconds'))
    return result


def species_data(species):
    rows = NATIVE[species]['rows']
    keep = U_PT[rows]
    return {'rows': rows[keep], 'counts': NATIVE[species]['counts'][keep], 'library': LIBRARY[rows][keep],
            'position': POSITION[rows][keep], 'specimen': SPECIMEN[rows][keep]}


D_ = {s: species_data(s) for s in SPECIES}
CALLS, Z86 = {}, {}
for s in SPECIES:
    payload = stored(T86, f'w_{s}_pt_shared')
    CALLS[s] = pd.read_csv(T86 / 'tables' / f'w_calls_{s}.csv', index_col=0)
    det = np.array([np.asarray((D_[s]['counts'][D_[s]['specimen'] == n] > 0).mean(axis=0)).ravel() for n in SPECIES[s]])
    expressed = (det >= .02).all(axis=0) & ~PANEL[s].str.upper().str.startswith('MT-')
    assert list(PANEL[s][expressed]) == list(CALLS[s].index), f'G5: {s} PT-expressed genes differ from 86'
    assert len(payload['alpha']) == len(CALLS[s])
    Z86[s] = {k: {n: payload[f'{k}|{n}'] for n in SPECIES[s]} for k in ('shape', 'centred')}
cols = PANEL['mouse'].get_indexer(CALLS['mouse'].index[:200])
again = no.species_zonation(D_['mouse']['counts'][:, cols], D_['mouse']['library'], D_['mouse']['position'],
                            D_['mouse']['specimen'], KNOTS, GRID, shapes=Z86['mouse']['shape'], n_jobs=N_JOBS)
for n in MOUSE:
    np.testing.assert_allclose(again['centred'][n], Z86['mouse']['centred'][n][:200], atol=1e-8, equal_nan=True)
GUARDS.append("G5: 86's cached W runs list its PT-expressed genes; 200 refitted mouse genes reproduce their curves")
print(GUARDS[-1])

CROSS = np.logical_and.reduce([np.isfinite(Z86[s]['centred'][n]).all(axis=0) for s in SPECIES for n in SPECIES[s]])
print(f'{int(CROSS.sum())} grid points covered by all four specimens (curve position {GRID[CROSS][0]:.2f}–{GRID[CROSS][-1]:.2f})')


def cross_curves(z):
    """Specimens' mean centred curve on CROSS for every column of a W payload's centred dict."""
    return np.mean([v[:, CROSS] for v in z.values()], axis=0)


CUR = {s: pd.DataFrame(cross_curves(Z86[s]['centred']), index=CALLS[s].index) for s in SPECIES}
STD = {s: pd.DataFrame(pz.standardize_rows(CUR[s].to_numpy()), index=CUR[s].index) for s in SPECIES}


def r_of(a, b):
    return float(pz.rowwise_pearson(np.atleast_2d(a), np.atleast_2d(b))[0])


def d_of(a, b):
    a, b = np.asarray(a, float), np.asarray(b, float)
    return float(np.sqrt(np.mean(((a - a.mean()) - (b - b.mean())) ** 2)))


def sums_fit(species, sets, stage):
    """W curves (86's fixed shapes) of summed counts: sets is a list of gene lists; returns CROSS curves (k x grid)."""
    idx = [PANEL[species].get_indexer(g) for g in sets]
    rows = np.concatenate([np.full(len(i), k) for k, i in enumerate(idx)])
    member = sparse.csr_matrix((np.ones(len(rows)), (np.concatenate(idx), rows)), shape=(len(PANEL[species]), len(sets)))
    x = sparse.csr_matrix(D_[species]['counts'] @ member)
    d = D_[species]
    payload = cached(stage, lambda: {f'c|{n}': v for n, v in no.species_zonation(
        x, d['library'], d['position'], d['specimen'], KNOTS, GRID, shapes=Z86[species]['shape'], n_jobs=N_JOBS)['centred'].items()},
        params={'knots': KNOTS, 'grid': GRID}, inputs={'counts': digest(x), 'shape': np.vstack(list(Z86[species]['shape'].values()))})
    return cross_curves({n: payload[f'c|{n}'] for n in SPECIES[species]})


# Strata (P1, P3) and expression deciles (P2)
STRATA, DECILE, ELIG = {}, {}, {}
for s in SPECIES:
    C = CALLS[s]
    clean = C.probe_clean.astype(bool)
    e_edges = np.quantile(C.log2_cp10k[clean], [.2, .4, .6, .8])
    z_edges = np.quantile(C.Z[clean], [.2, .4, .6, .8])
    STRATA[s] = pd.Series(np.digitize(C.log2_cp10k, e_edges) * 5 + np.digitize(C.Z, z_edges), index=C.index)
    ELIG[s] = C.index[clean]
    d = D_[s]
    cp = np.mean([np.asarray(d['counts'][d['specimen'] == n].sum(axis=0)).ravel() / d['library'][d['specimen'] == n].sum() * 1e4
                  for n in SPECIES[s]], axis=0)
    lg = pd.Series(np.log2(cp + .01), index=PANEL[s])
    DECILE[s] = pd.Series(pd.qcut(lg.rank(method='first'), 10, labels=False).to_numpy(), index=PANEL[s])
POOL_P2 = {s: INV[s].index[INV[s].probe_clean.astype(bool) & ~INV[s].index.str.upper().str.startswith('MT-')] for s in SPECIES}
rng = np.random.default_rng(SEED)


def draw_like(s, genes, comp, how, n_draws):
    """n_draws x len(genes) random replacements: same stratum (how='strata', PT-expressed probe-clean) or expression
    decile (how='decile', probe-clean panel genes), outside orthogroup ``comp``."""
    outside = INV[s].component.ne(comp)
    if how == 'strata':
        pool = pd.Index([g for g in ELIG[s] if outside[g]])
        key, keys = STRATA[s], STRATA[s].reindex(pool)
    else:
        pool = pd.Index([g for g in POOL_P2[s] if outside[g]])
        key, keys = DECILE[s], DECILE[s].reindex(pool)
    out = np.empty((n_draws, len(genes)), dtype=object)
    for j, g in enumerate(genes):
        cand = pool[keys.to_numpy() == key[g]]
        if not len(cand):
            raise ValueError(f'No replacement candidate for {g}')
        out[:, j] = rng.choice(cand.to_numpy(), n_draws)
    return out

## 4 · The 71-called kept pairs inside multi-member orthogroups

In [ ]:
hs = UNITS.var.human_symbol.astype(str)
q71 = STAT71['condition'].q
pairs = []
for m, h in hs.items():
    if not (INV['mouse'].class_support3.get(m) in NON or INV['human'].class_support3.get(h) in NON):
        continue
    if m not in CALLS['mouse'].index or h not in CALLS['human'].index:
        continue
    if not (CALLS['mouse'].loc[m, 'probe_clean'] and CALLS['human'].loc[h, 'probe_clean']) or not q71.get(m, 1) <= ALPHA:
        continue
    comp = int(INV['mouse'].component[m])
    assert int(INV['human'].component[h]) == comp, (m, h)
    pairs.append({'mouse': m, 'human': h, 'family': comp, 'q71': float(q71[m])})
PAIRS = pd.DataFrame(pairs)


def others(comp, s, exclude):
    g = INV[s].index[INV[s].component.eq(comp)]
    return [x for x in g if x != exclude and x in CALLS[s].index and bool(CALLS[s].loc[x, 'probe_clean'])]


PAIRS['mouse alternatives'] = [others(r.family, 'mouse', r.mouse) for r in PAIRS.itertuples()]
PAIRS['human alternatives'] = [others(r.family, 'human', r.human) for r in PAIRS.itertuples()]
PAIRS['r kept'] = [r_of(CUR['mouse'].loc[r.mouse], CUR['human'].loc[r.human]) for r in PAIRS.itertuples()]
PAIRS['D kept'] = [d_of(CUR['mouse'].loc[r.mouse], CUR['human'].loc[r.human]) for r in PAIRS.itertuples()]
print(f"{len(PAIRS)} 71-called kept pairs in multi-member orthogroups; {int((PAIRS['mouse alternatives'].map(len) > 0).sum())} "
      f"with a mouse alternative, {int((PAIRS['human alternatives'].map(len) > 0).sum())} with a human alternative")

## 5 · P1 · Substitution: another member carries the profile the kept partner lacks

In [ ]:
def best_gain(anchor_std, kept_std, alt_std):
    """(gain, r_alt, index) of the best alternative for one anchor."""
    r_alt = alt_std @ anchor_std
    k = int(np.argmax(r_alt))
    return float(r_alt[k] - kept_std @ anchor_std), float(r_alt[k]), k


rows1, P1_UNITS = [], []
for idx, r in PAIRS.iterrows():
    best = None
    for side, anchor_s, anchor, kept, alts in (('mouse', 'human', r.human, r.mouse, r['mouse alternatives']),
                                               ('human', 'mouse', r.mouse, r.human, r['human alternatives'])):
        if not alts:
            continue
        a_std = STD[anchor_s].loc[anchor].to_numpy()
        g, ra, k = best_gain(a_std, STD[side].loc[kept].to_numpy(), STD[side].loc[alts].to_numpy())
        P1_UNITS.append((idx, side, anchor_s, anchor, kept, alts))
        if best is None or g > best['gain']:
            best = {'gain': g, 'r alt': ra, 'substituting member': alts[k], 'side': side}
    if best is not None:
        rows1.append({'mouse': r.mouse, 'human': r.human, 'family': r.family, 'q71': r.q71, 'r kept': r['r kept'], **best})
P1 = pd.DataFrame(rows1)
P1['substituted'] = (P1.gain >= GAIN) & (P1['r alt'] >= R_ALT)
obs1 = int(P1.substituted.sum())

null_gain = {i: np.full(N_NULL_P1, -np.inf) for i in P1.index}
null_ra = {i: np.full(N_NULL_P1, -np.inf) for i in P1.index}
row_of = {(r.mouse, r.human): i for i, r in P1.iterrows()}
for idx, side, anchor_s, anchor, kept, alts in P1_UNITS:
    rr = PAIRS.loc[idx]
    i = row_of[(rr.mouse, rr.human)]
    a_std = STD[anchor_s].loc[anchor].to_numpy()
    kept_r = STD[side].loc[kept].to_numpy() @ a_std
    draws = draw_like(side, alts, rr.family, 'strata', N_NULL_P1)
    r_draw = (STD[side].loc[draws.ravel()].to_numpy() @ a_std).reshape(draws.shape)
    best_r = r_draw.max(axis=1)
    better = best_r - kept_r > null_gain[i]
    null_gain[i] = np.where(better, best_r - kept_r, null_gain[i])
    null_ra[i] = np.where(better, best_r, null_ra[i])
null_counts = np.sum([(null_gain[i] >= GAIN) & (null_ra[i] >= R_ALT) for i in P1.index], axis=0)
p1 = (1 + int((null_counts >= obs1).sum())) / (1 + N_NULL_P1)
P1['gain p (empirical)'] = [(1 + int((null_gain[i] >= P1.loc[i, 'gain']).sum())) / (1 + N_NULL_P1) for i in P1.index]
P1['anchor gene'] = np.where(P1.side.eq('mouse'), P1.human, P1.mouse)
P1['kept partner'] = np.where(P1.side.eq('mouse'), P1.mouse, P1.human)
P1['partner peak'] = [CALLS[s].loc[g, 'peak'] for s, g in zip(P1.side, P1['kept partner'])]
P1['substitute peak'] = [CALLS[s].loc[g, 'peak'] for s, g in zip(P1.side, P1['substituting member'])]
P1['substitute zonated'] = [bool(CALLS[s].loc[g, 'zonated']) for s, g in zip(P1.side, P1['substituting member'])]
print(f'P1: {obs1} of {len(P1)} pairs substituted; null mean {null_counts.mean():.2f} (95% {np.percentile(null_counts, 95):.0f}); '
      f'one-sided p {p1:.4f}')
primary('P1: substitution by another paralog (count against matched random alternatives)', p1,
        f'{obs1}/{len(P1)} vs null mean {null_counts.mean():.2f}')
display(P1.sort_values('gain', ascending=False).round(3))
P1.to_csv(tables / 'p1_substitution.csv', index=False)
pd.Series(null_counts).to_csv(tables / 'p1_null_counts.csv', index=False, header=['substituted'])

## 6 · P2 · Compensation: do the family sums agree where the kept pair differs?

In [ ]:
def family_members(comp, s):
    m = INV[s][(INV[s].component == comp) & INV[s].family_clean.astype(bool)]
    return [g for g in m.index if not g.upper().startswith('MT-')]


P2 = PAIRS[['mouse', 'human', 'family', 'q71', 'r kept', 'D kept']].copy()
P2['mouse members'] = [family_members(c, 'mouse') for c in P2.family]
P2['human members'] = [family_members(c, 'human') for c in P2.family]
assert all(r.mouse in r['mouse members'] and r.human in r['human members'] for _, r in P2.iterrows()), 'a kept gene is not family-clean'
P2 = P2[(P2['mouse members'].map(len) > 1) | (P2['human members'].map(len) > 1)].reset_index(drop=True)
print(f'P2: {len(P2)} pairs whose family holds more than the kept gene in at least one species')
SUM_OBS = {s: sums_fit(s, list(P2[f'{s} members']), f'p2_family_sums_{s}') for s in SPECIES}
P2['D family'] = [d_of(SUM_OBS['mouse'][k], SUM_OBS['human'][k]) for k in range(len(P2))]
P2['r family'] = [r_of(SUM_OBS['mouse'][k], SUM_OBS['human'][k]) for k in range(len(P2))]
P2['L'] = np.log2(P2['D family'] / P2['D kept'])

null_sets = {s: [] for s in SPECIES}
for _, r in P2.iterrows():
    for s, kept, members in (('mouse', r.mouse, r['mouse members']), ('human', r.human, r['human members'])):
        rest = [g for g in members if g != kept]
        draws = draw_like(s, rest, r.family, 'decile', N_NULL_SUMS) if rest else np.empty((N_NULL_SUMS, 0), dtype=object)
        null_sets[s].extend([[kept, *d] for d in draws])
SUM_NULL = {s: sums_fit(s, null_sets[s], f'p2_null_sums_{s}') for s in SPECIES}
L_null = np.array([[np.log2(d_of(SUM_NULL['mouse'][k * N_NULL_SUMS + d], SUM_NULL['human'][k * N_NULL_SUMS + d]) / P2.loc[k, 'D kept'])
                    for d in range(N_NULL_SUMS)] for k in range(len(P2))])
P2['percentile'] = (L_null <= P2.L.to_numpy()[:, None]).mean(axis=1)
P2['null L median'] = np.median(L_null, axis=1)
P2['accounted for'] = P2.L <= L_ACCOUNTED
w2 = wilcoxon(P2.percentile - .5, alternative='less')
null_acc = (L_null <= L_ACCOUNTED).sum(axis=0)
print(f"P2: median L {P2.L.median():.2f} (null median {np.median(L_null):.2f}); median percentile {P2.percentile.median():.2f}; "
      f"Wilcoxon one-sided p {w2.pvalue:.3g}. Accounted for (L ≤ −1): {int(P2['accounted for'].sum())} of {len(P2)}, "
      f"null mean {null_acc.mean():.2f}")
primary('P2: family sums agree better than matched pseudo-families (Wilcoxon on percentiles)', w2.pvalue,
        f'median percentile {P2.percentile.median():.2f}')
show2 = P2.assign(**{'mouse members': P2['mouse members'].map(', '.join), 'human members': P2['human members'].map(', '.join)})
display(show2.sort_values('percentile').round(3).head(40))
show2.to_csv(tables / 'p2_compensation.csv', index=False)

## 7 · P3 · Division of labour: does one gene span what several genes split?

In [ ]:
comps = sorted(set(INV['mouse'].component[INV['mouse'].non]) | set(INV['human'].component[INV['human'].non]))
anchors = []
for comp in comps:
    for A, B in (('mouse', 'human'), ('human', 'mouse')):
        e = [g for g in family_members(comp, B) if g in CALLS[B].index]
        if len(e) < 2:
            continue
        for g in INV[A].index[INV[A].component.eq(comp)]:
            if g in CALLS[A].index and CALLS[A].loc[g, 'probe_clean'] and CALLS[A].loc[g, 'zonated']:
                anchors.append({'family': int(comp), 'anchor species': A, 'anchor': g, 'members': e})
P3 = pd.DataFrame(anchors)
esets = {B: [] for B in SPECIES}
pos3 = {}
for a in anchors:
    B = 'human' if a['anchor species'] == 'mouse' else 'mouse'
    key = (B, a['family'])
    if key not in pos3:
        pos3[key] = len(esets[B])
        esets[B].append(a['members'])
SUM3 = {B: sums_fit(B, esets[B], f'p3_sums_{B}') for B in SPECIES if esets[B]}
rows3 = []
for a in anchors:
    B = 'human' if a['anchor species'] == 'mouse' else 'mouse'
    s_curve = CUR[a['anchor species']].loc[a['anchor']].to_numpy()
    r_members = [r_of(s_curve, CUR[B].loc[g]) for g in a['members']]
    r_sum = r_of(s_curve, SUM3[B][pos3[(B, a['family'])]])
    rows3.append({**{k: v for k, v in a.items() if k != 'members'}, 'members': ', '.join(a['members']),
                  'best member': a['members'][int(np.argmax(r_members))], 'r best member': max(r_members), 'r sum': r_sum,
                  'delta': r_sum - max(r_members)})
P3 = pd.DataFrame(rows3)
P3['sum beats best'] = P3.delta > 0
obs3 = int(P3['sum beats best'].sum())
null3_sets = {B: [] for B in SPECIES}
null3_pos = {}
for key, k in pos3.items():
    B, comp = key
    members = esets[B][k]
    draws = draw_like(B, members, comp, 'strata', N_NULL_SUMS)
    null3_pos[key] = len(null3_sets[B])
    null3_sets[B].extend([list(d) for d in draws])
SUM3N = {B: sums_fit(B, null3_sets[B], f'p3_null_sums_{B}') for B in SPECIES if null3_sets[B]}
null3 = np.zeros(N_NULL_SUMS, int)
for a in anchors:
    B = 'human' if a['anchor species'] == 'mouse' else 'mouse'
    s_curve = CUR[a['anchor species']].loc[a['anchor']].to_numpy()
    start = null3_pos[(B, a['family'])]
    draws_curves = SUM3N[B][start:start + N_NULL_SUMS]
    # the null members' own curves: random genes are PT-expressed, so their curves exist
    for d in range(N_NULL_SUMS):
        genes_d = null3_sets[B][start + d]
        r_m = max(r_of(s_curve, CUR[B].loc[g]) for g in genes_d)
        null3[d] += int(r_of(s_curve, draws_curves[d]) > r_m)
p3 = (1 + int((null3 >= obs3).sum())) / (1 + N_NULL_SUMS)
print(f'P3: the family sum beats the best member for {obs3} of {len(P3)} anchors; null mean {null3.mean():.2f}; one-sided p {p3:.4f}')
primary('P3: a family sum matches a single gene better than its best member (count against random sums)', p3,
        f'{obs3}/{len(P3)} vs null mean {null3.mean():.2f}')
display(P3.round(3))
P3.to_csv(tables / 'p3_division_of_labour.csv', index=False)

## 8 · P4–P6 (descriptive)

In [ ]:
from scipy.optimize import linear_sum_assignment

hm = pd.read_csv(COMPARA_HM, sep='\t', usecols=['gene_stable_id', 'homology_gene_stable_id', 'identity', 'homology_identity'])
hm['identity_mean'] = (hm.identity + hm.homology_identity) / 2
hm_id = hm.groupby(['gene_stable_id', 'homology_gene_stable_id']).identity_mean.max()
gid = {s: INV[s].gene_id.astype(str).str.split('.').str[0] for s in SPECIES}
rows4 = []
for comp in comps:
    mm = others(comp, 'mouse', None)
    hh = others(comp, 'human', None)
    if len(mm) < 2 or len(hh) < 2 or not (CALLS['mouse'].reindex(mm).zonated.any() or CALLS['human'].reindex(hh).zonated.any()):
        continue
    R = STD['mouse'].loc[mm].to_numpy() @ STD['human'].loc[hh].to_numpy().T
    I = np.array([[hm_id.get((gid['human'][h], gid['mouse'][m]), 0.) for h in hh] for m in mm])
    pr, pc = linear_sum_assignment(-R)
    sr, sc_ = linear_sum_assignment(-I)
    pos_pairs, seq_pairs = set(zip(pr, pc)), set(zip(sr, sc_))
    perm = []
    for _ in range(500):
        rows_p, cols_p = rng.permutation(len(mm))[:len(pr)], rng.permutation(len(hh))[:len(pr)]
        perm.append(len(set(zip(rows_p, cols_p)) & seq_pairs))
    rows4.append({'family': int(comp), 'mouse': ', '.join(mm), 'human': ', '.join(hh), 'pairs': len(pos_pairs),
                  'positional = sequence': len(pos_pairs & seq_pairs), 'chance (mean)': float(np.mean(perm)),
                  'positional pairs': ', '.join(f'{mm[i]}–{hh[j]} ({R[i, j]:.2f})' for i, j in sorted(pos_pairs))})
P4 = pd.DataFrame(rows4)
if len(P4):
    print(f"P4: {len(P4)} families; positional = sequence in {int(P4['positional = sequence'].sum())} of {int(P4.pairs.sum())} pairs "
          f"(chance {P4['chance (mean)'].sum():.1f})")
    display(P4)
P4.to_csv(tables / 'p4_positional_vs_sequence.csv', index=False)

F86 = pd.read_csv(T86 / 'tables' / 'h7_family_units.csv')
B86 = pd.read_csv(T86 / 'tables' / 'h7b_one_to_one_calls_in_families.csv')
called_kept = set(B86.family[B86['gene called'].astype(bool)])
P5 = F86[F86.called.astype(bool) & (F86['p relabeling 1'] > ALPHA) & (F86['p relabeling 2'] > ALPHA) & ~F86.family.isin(called_kept)].copy()
for s in SPECIES:
    P5[f'max Z {s}'] = [CALLS[s][CALLS[s].component.eq(c)].Z.max() for c in P5.family]
    P5[f'zonated {s}'] = [', '.join(CALLS[s].index[CALLS[s].component.eq(c) & CALLS[s].zonated]) for c in P5.family]
print(f'P5: {len(P5)} family units called (relabeled p > 0.05 in both) with no 71-called kept pair')
display(P5[['label', 'q (joint)', 'max Z mouse', 'max Z human', 'zonated mouse', 'zonated human']].sort_values('q (joint)').round(3))
P5.to_csv(tables / 'p5_family_only_differences.csv', index=False)

rows6 = []
for s, other in (('mouse', 'human'), ('human', 'mouse')):
    C = CALLS[s]
    lone = C[C.strong & C.probe_clean.astype(bool) & C.class_support3.isin(['no_ortholog', 'no_consensus'])]
    O = STD[other].loc[ELIG[other]]
    for g in lone.index:
        rr = O.to_numpy() @ STD[s].loc[g].to_numpy()
        top = np.argsort(rr)[::-1][:5]
        rows6.append({'species': s, 'gene': g, 'class': lone.loc[g, 'class_support3'], 'Z': lone.loc[g, 'Z'],
                      'peak': lone.loc[g, 'peak'],
                      'closest in the other species': ', '.join(f'{O.index[k]} ({rr[k]:.2f}, Z {CALLS[other].loc[O.index[k], "Z"]:.1f})' for k in top)})
P6 = pd.DataFrame(rows6)
display(P6.round(2))
P6.to_csv(tables / 'p6_niches.csv', index=False)

## 9 · Multiplicity

In [ ]:
from scipy.stats import false_discovery_control

PRIM = pd.DataFrame(PRIMARY)
PRIM['BH over the three'] = false_discovery_control(PRIM.p.to_numpy())
display(PRIM.round(4))
PRIM.to_csv(tables / 'primary_tests.csv', index=False)

## 10 · Figures (plots only)

- **88a:** P1: the null count of substituted pairs with the observed count; the substituted pairs (the anchor in its
  species' colour, the kept partner dashed, the substituting member solid).
- **88b:** P2: D of the kept pair against D of the family sums (the diagonal and the halving line); per-family
  percentiles; the two families with the lowest percentile, kept pair dashed and family sums solid.
- **88c:** P3: the null count with the observed count; the two anchors with the largest δ.

In [ ]:
GX = GRID[CROSS]
COL = {'mouse': '#0072B2', 'human': '#D55E00'}


def null_panel(ax, null, observed):
    ax.hist(null, bins=np.arange(null.min(), max(null.max(), observed) + 2) - .5, color=GREY, lw=0)
    ax.axvline(observed, color=INK, lw=1)
    ax.set_xlabel('count')
    ax.set_ylabel('draws')


def fig_a():
    sub = P1[P1.substituted].sort_values('gain', ascending=False).head(4)
    fig, axes = plt.subplots(1, 1 + max(len(sub), 1), figsize=(40 * MM * (1 + max(len(sub), 1)), 42 * MM),
                             gridspec_kw={'wspace': .35})
    axes = np.atleast_1d(axes)
    null_panel(axes[0], null_counts, obs1)
    for ax, (_, r) in zip(axes[1:], sub.iterrows()):
        anchor_s = 'human' if r.side == 'mouse' else 'mouse'
        ax.plot(GX, CUR[anchor_s].loc[r['anchor gene']], color=COL[anchor_s])
        ax.plot(GX, CUR[r.side].loc[r['kept partner']], color=COL[r.side], ls='--')
        ax.plot(GX, CUR[r.side].loc[r['substituting member']], color=COL[r.side])
        ax.set_title(f"{r['anchor gene']}\n{r['kept partner']} / {r['substituting member']}", loc='left')
        ax.set_xlabel('curve position')
    for ax in axes[1 + len(sub):]:
        ax.axis('off')
    save(fig, 'fig88a_substitution')


def fig_b():
    ex = P2.sort_values('percentile').head(2)
    fig, axes = plt.subplots(1, 4, figsize=(175 * MM, 44 * MM), gridspec_kw={'wspace': .4})
    ax = axes[0]
    ax.scatter(P2['D kept'], P2['D family'], s=6, color=INK, lw=0)
    lim = [0, max(P2['D kept'].max(), P2['D family'].max()) * 1.05]
    ax.plot(lim, lim, color=LIGHT, lw=.6)
    ax.plot(lim, [v / 2 for v in lim], color=LIGHT, lw=.6, ls='--')
    ax.set_xlabel('D kept pair')
    ax.set_ylabel('D family sums')
    ax = axes[1]
    ax.hist(P2.percentile, bins=np.linspace(0, 1, 11), color=GREY, lw=0)
    ax.axvline(.5, color=INK, lw=.6, ls=':')
    ax.set_xlabel('percentile among pseudo-families')
    for ax, (k, r) in zip(axes[2:], ex.iterrows()):
        for s, g in (('mouse', r.mouse), ('human', r.human)):
            ax.plot(GX, CUR[s].loc[g], color=COL[s], ls='--')
            ax.plot(GX, SUM_OBS[s][k], color=COL[s])
        ax.set_title(f'{r.mouse} / {r.human}', loc='left')
        ax.set_xlabel('curve position')
    save(fig, 'fig88b_compensation')


def fig_c():
    ex = P3.sort_values('delta', ascending=False).head(2)
    fig, axes = plt.subplots(1, 3, figsize=(150 * MM, 44 * MM), gridspec_kw={'wspace': .35})
    null_panel(axes[0], null3, obs3)
    for ax, (_, r) in zip(axes[1:], ex.iterrows()):
        A = r['anchor species']
        B = 'human' if A == 'mouse' else 'mouse'
        ax.plot(GX, CUR[A].loc[r.anchor], color=COL[A], lw=1.4)
        for g in r.members.split(', '):
            ax.plot(GX, CUR[B].loc[g], color=COL[B], lw=.6, alpha=.7)
        ax.plot(GX, SUM3[B][pos3[(B, r.family)]], color=COL[B], lw=1.4, ls='--')
        ax.set_title(r.anchor, loc='left')
        ax.set_xlabel('curve position')
    save(fig, 'fig88c_division_of_labour')


for f in (fig_a, fig_b, fig_c):
    f()

## 11 · Run record

In [ ]:
import platform
from importlib.metadata import version

record = {'notebook_logic_version': NOTEBOOK_LOGIC_VERSION, 'protocol': 'docs/results/pt-paralog-substitution.md',
          'guards': GUARDS, 'external_sha256': EXTERNAL_SHA, 'pairs': len(PAIRS), 'p1_pairs': len(P1), 'p2_families': len(P2),
          'p3_anchors': len(P3), 'primary_tests': PRIM.to_dict(orient='records'), 'seed': SEED, 'stage_seconds': SECONDS,
          'versions': {'python': platform.python_version(), **{m_: version(m_) for m_ in ('numpy', 'scipy', 'pandas')}},
          'wall_seconds': time.perf_counter() - NOTEBOOK_START}
(output / 'run_record.json').write_text(json.dumps(record, indent=2, default=str))
print(f"Wall time {record['wall_seconds']:.0f} s")